In [2]:
"""
Graph-structure fidelity metrics for Berka-style transaction tables
(real vs. ClavaDDPM-synthetic).

Usage
-----
    real_trans = pd.read_csv(REAL_TRANS_PATH)
    synthetic_trans = pd.read_csv(SYNTHETIC_TRANS_PATH)
    results = evaluate(real_trans, synthetic_trans)

Only needs the `trans` table as ClavaDDPM outputs it (account_id, date, amount,
type, operation, k_symbol, bank, account). No upstream changes are required;
if the synthetic counterparty identities are not reused, the metrics simply
show that gap.

Terminology: a "counterparty" is identified by "bank:account" of the other side of a
transaction (rows whose counterparty account is a missing value have none). An "edge" is a
transaction with a counterparty; a "pair" is an (account, counterparty) combination;
a pair with several transactions is a "repeated" pair (recurring payment).

Metric groups (all isomorphism-invariant: no node matching is needed, and they
work with or without communities). Output metric names use the abbreviation "cp" for
counterparty:

  presence          counterparty-present rate, per-account share, rate by type/operation/k_symbol
  account           per-account degree / repeat / concentration features
  pair              pair multiplicity (transactions per account-counterparty pair)
  counterparty      counterparty in-degree (distinct accounts) and volume, hub concentration
  edge_consistency  attribute stability inside repeated pairs
  structure         giant/dyad components; wedges + butterflies (+ null z) only if the real
                    data has shared counterparties (min_shared_counterparty_share_for_motifs)
  bank              coarse account x bank graph: bank mix, banks per account, bank co-usage
  temporal          inter-event gaps per pair, gap regularity, monthly recurrence
  coupling          amount/type/operation/k_symbol/bank conditioned on pair multiplicity
                    and counterparty in-degree
  c2st              classifier two-sample test on account-level graph features

Every distance is reported next to a noise floor from real-vs-real (random split of
the real accounts into two halves). ratio = distance / floor; ratio <= GAP_RATIO_THRESHOLD
-> "ok" (heuristic). If the number of accounts differs by more than 5%, the larger side is
subsampled by accounts so scale-dependent metrics are comparable.

Repeat-dependent metrics (edge_consistency, temporal) need at least MIN_SAMPLE_SIZE repeated
pairs / gaps on a side; with fewer they count as "absent" (distance 1.0 if the other side has them).

Distances: KS for numeric distributions, total variation for categorical,
absolute difference for rates, |log ratio| for unbounded counts.
"""
import numpy as np
import pandas as pd
from scipy import sparse
from scipy.sparse.csgraph import connected_components
from scipy.stats import ks_2samp
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict

# --------------------------------------------------------------------------- #
# Configuration
# --------------------------------------------------------------------------- #
REAL_TRANS_PATH = "real_trans.csv"
SYNTHETIC_TRANS_PATH = "synthetic_trans.csv"
OUT_PATH = "graph_fidelity_results.csv"

# Maps the names used in this module to the column names of your tables.
COLUMN_MAP = dict(
    account_id="account_id",
    date="trans_date",
    amount="amount",
    trans_type="trans_type",
    operation="operation",
    k_symbol="k_symbol",
    counterparty_bank="bank",
    counterparty_account="account",
    # "days": numeric day offsets (ClavaDDPM-encoded dates; floats are floored).
    # "%y%m%d": Berka YYMMDD integers. None: datetime column or ISO string.
    date_format="days",
    # Values of the counterparty account column that mean "no counterparty".
    counterparty_missing_values=("0",),
)

MISSING_STRINGS = {"", "nan", "none", "<na>", "null", "nat"}
LOG_RATIO_EPSILON = 1e-6        # added to both sides of a log ratio to avoid log(0)
MIN_SAMPLE_SIZE = 30            # minimum rows / repeated pairs / gaps for a comparison
MIN_NOISE_FLOOR = 1e-3          # noise floors below this are clipped when forming the ratio
GAP_RATIO_THRESHOLD = 2         # ratio above this is flagged "GAP"

# Account-level features fed to the classifier two-sample test.
CLASSIFIER_FEATURES = [
    "n_transactions", "n_counterparty_transactions", "counterparty_share",
    "n_distinct_counterparties", "repeat_rate", "max_pair_multiplicity",
    "counterparty_entropy", "mean_counterparty_indegree", "frac_shared_counterparties",
    "n_banks", "bank_entropy", "median_gap_days",
]

# Output metric name -> account-feature column (the "account" metric group).
ACCOUNT_METRIC_COLUMNS = {
    "n_distinct_cp": "n_distinct_counterparties",
    "repeat_rate": "repeat_rate",
    "cp_entropy": "counterparty_entropy",
    "frac_shared_cp": "frac_shared_counterparties",
}

# Output labels -> internal column names for the conditional ("coupling") metrics.
PRESENCE_CATEGORIES = {"type": "trans_type", "operation": "operation", "k_symbol": "k_symbol"}
COUPLING_BUCKETINGS = {"edge_mult": "pair_multiplicity", "cp_indeg": "counterparty_indegree"}
COUPLING_VARIABLES = {
    "log_amt": "log_amount", "type": "trans_type", "operation": "operation",
    "k_symbol": "k_symbol", "bank": "counterparty_bank",
}


# --------------------------------------------------------------------------- #
# Preparation
# --------------------------------------------------------------------------- #
def _clean_id_column(series):
    """String ids with missing values (NaN, 'nan', '', ...) mapped to NaN."""
    series = series.astype("object")
    is_na = series.isna()
    cleaned = (series.where(~is_na, "nan").astype(str).str.strip()
               .str.replace(r"\.0+$", "", regex=True))
    is_missing = is_na | cleaned.str.lower().isin(MISSING_STRINGS)
    return cleaned.astype("object").mask(is_missing)


def _parse_dates(series, date_format):
    if pd.api.types.is_datetime64_any_dtype(series):
        return series
    if date_format == "days":
        days = np.floor(pd.to_numeric(series, errors="coerce"))
        return pd.Timestamp("2000-01-01") + pd.to_timedelta(days, unit="D")
    if date_format is None:
        return pd.to_datetime(series, errors="coerce")
    return pd.to_datetime(_clean_id_column(series), format=date_format, errors="coerce")


def clean_transactions(trans, column_map):
    """Standardised copy of the transaction table with internal column names."""
    out = pd.DataFrame(index=trans.index)
    out["account"] = _clean_id_column(trans[column_map["account_id"]])

    counterparty_account = _clean_id_column(trans[column_map["counterparty_account"]])
    counterparty_account = counterparty_account.mask(
        counterparty_account.isin(column_map.get("counterparty_missing_values", ())))
    bank = _clean_id_column(trans[column_map["counterparty_bank"]])

    out["counterparty_account_number"] = counterparty_account
    out["has_counterparty"] = counterparty_account.notna()
    out["counterparty_bank"] = bank.fillna("?").where(out["has_counterparty"])
    out["counterparty"] = (bank.fillna("?") + ":" + counterparty_account).where(
        out["has_counterparty"])

    amount = pd.to_numeric(trans[column_map["amount"]], errors="coerce")
    out["log_amount"] = np.log1p(amount.abs())
    for name in ("trans_type", "operation", "k_symbol"):
        out[name] = _clean_id_column(trans[column_map[name]]).fillna("NA")
    out["date"] = _parse_dates(trans[column_map["date"]], column_map["date_format"])
    return out[out["account"].notna()].copy()


def match_account_counts(real_trans, synthetic_trans, column_map, seed=0):
    """Subsample accounts of the larger table when the account counts differ by > 5%."""
    n_real = _clean_id_column(real_trans[column_map["account_id"]]).nunique()
    n_synthetic = _clean_id_column(synthetic_trans[column_map["account_id"]]).nunique()
    if abs(n_real - n_synthetic) / max(n_real, n_synthetic) <= 0.05:
        return real_trans, synthetic_trans
    n_keep = min(n_real, n_synthetic)
    rng = np.random.default_rng(seed)

    def subsample_accounts(trans):
        account_ids = _clean_id_column(trans[column_map["account_id"]])
        kept = rng.choice(account_ids.dropna().unique(), size=n_keep, replace=False)
        return trans[account_ids.isin(set(kept))]

    if n_real > n_synthetic:
        return subsample_accounts(real_trans), synthetic_trans
    return real_trans, subsample_accounts(synthetic_trans)


def split_accounts_in_half(trans, column_map, seed):
    """Random split of the accounts into two disjoint halves (for the noise floor)."""
    account_ids = _clean_id_column(trans[column_map["account_id"]])
    unique_accounts = account_ids.dropna().unique()
    rng = np.random.default_rng(seed)
    rng.shuffle(unique_accounts)
    first_half = set(unique_accounts[: len(unique_accounts) // 2])
    in_first_half = account_ids.isin(first_half)
    return trans[in_first_half], trans[~in_first_half & account_ids.notna()]


# --------------------------------------------------------------------------- #
# Graph helpers
# --------------------------------------------------------------------------- #
def bipartite_adjacency(account_idx, counterparty_idx, n_accounts, n_counterparties):
    """Accounts x counterparties sparse matrix (entry = 1 per distinct pair)."""
    return sparse.csr_matrix(
        (np.ones(len(account_idx), dtype=np.int64), (account_idx, counterparty_idx)),
        shape=(n_accounts, n_counterparties),
    )


def connected_component_stats(adjacency):
    n_accounts, n_counterparties = adjacency.shape
    full_graph = sparse.bmat([[None, adjacency], [adjacency.T, None]], format="csr")
    _, node_component = connected_components(full_graph, directed=False)
    component_sizes = np.bincount(node_component)
    return dict(
        giant_component_fraction=component_sizes.max() / (n_accounts + n_counterparties),
        dyad_component_share=float((component_sizes == 2).mean()),
    )


def count_butterflies_and_wedges(adjacency):
    """Number of 4-cycles (K_{2,2}) and of wedges centred on counterparties."""
    adjacency = adjacency.tocsr()
    counterparty_degree = np.asarray(adjacency.sum(0)).ravel()
    account_degree = np.asarray(adjacency.sum(1)).ravel()
    # account pairs sharing a counterparty
    wedges_via_counterparties = float(
        (counterparty_degree * (counterparty_degree - 1) / 2).sum())
    wedges_via_accounts = float((account_degree * (account_degree - 1) / 2).sum())
    # count shared neighbours on the side that has fewer wedges (cheaper product)
    if wedges_via_counterparties <= wedges_via_accounts:
        shared_neighbours = adjacency @ adjacency.T
    else:
        shared_neighbours = adjacency.T @ adjacency
    shared_neighbours = sparse.triu(shared_neighbours, k=1)
    pair_counts = shared_neighbours.data.astype(np.float64)
    n_butterflies = float((pair_counts * (pair_counts - 1) / 2).sum())
    return n_butterflies, wedges_via_counterparties


def degree_preserving_edge_swaps(account_idx, counterparty_idx, n_swaps, rng):
    """Randomise a simple bipartite graph with double edge swaps (degrees preserved)."""
    n_counterparties = int(counterparty_idx.max()) + 1
    accounts = account_idx.tolist()
    counterparties = counterparty_idx.tolist()
    existing_edges = {a * n_counterparties + c for a, c in zip(accounts, counterparties)}
    candidate_pairs = rng.integers(0, len(accounts), size=(n_swaps, 2)).tolist()
    for i, j in candidate_pairs:
        account_i, counterparty_i = accounts[i], counterparties[i]
        account_j, counterparty_j = accounts[j], counterparties[j]
        if account_i == account_j or counterparty_i == counterparty_j:
            continue
        new_edge_i = account_i * n_counterparties + counterparty_j
        new_edge_j = account_j * n_counterparties + counterparty_i
        if new_edge_i in existing_edges or new_edge_j in existing_edges:
            continue
        existing_edges.remove(account_i * n_counterparties + counterparty_i)
        existing_edges.remove(account_j * n_counterparties + counterparty_j)
        existing_edges.add(new_edge_i)
        existing_edges.add(new_edge_j)
        counterparties[i], counterparties[j] = counterparty_j, counterparty_i
    return np.asarray(counterparties, dtype=np.int64)


# --------------------------------------------------------------------------- #
# Per-dataset bundle of metrics
# --------------------------------------------------------------------------- #
def build_metric_bundle(trans, column_map):
    """Everything needed to compare one table: cleaned data, per-account features and
    the single-table metrics (`bundle["metrics"]`: (group, name) -> (kind, payload))."""
    transactions = clean_transactions(trans, column_map)
    edges = (transactions[transactions["has_counterparty"]]
             .sort_values("date", kind="stable").copy())
    if edges.empty:
        raise ValueError("No transactions with a counterparty.")
    pair_keys = ["account", "counterparty"]

    edges["pair_multiplicity"] = edges.groupby(pair_keys, sort=False)["counterparty"].transform("size")
    counterparty_stats = edges.groupby("counterparty").agg(
        n_transactions=("counterparty", "size"), indegree=("account", "nunique"))
    edges["counterparty_indegree"] = edges["counterparty"].map(counterparty_stats["indegree"])
    edges["is_repeat_payment"] = edges.groupby(pair_keys, sort=False).cumcount() > 0
    edges["days_since_previous_payment"] = (
        edges.groupby(pair_keys, sort=False)["date"].diff().dt.days)

    pairs = edges.groupby(pair_keys, sort=False).size().rename("n_payments").reset_index()
    pairs = pairs.merge(counterparty_stats["indegree"], left_on="counterparty", right_index=True)
    pairs["counterparty_is_shared"] = pairs["indegree"] > 1
    payment_share = pairs["n_payments"] / pairs.groupby("account")["n_payments"].transform("sum")
    pairs["entropy_term"] = -payment_share * np.log(payment_share)

    # account feature table ---------------------------------------------------
    accounts = pd.DataFrame(index=pd.Index(transactions["account"].unique(), name="account"))
    accounts["n_transactions"] = transactions.groupby("account").size()
    accounts["n_counterparty_transactions"] = (
        edges.groupby("account").size().reindex(accounts.index).fillna(0))
    accounts["counterparty_share"] = (
        accounts["n_counterparty_transactions"] / accounts["n_transactions"])
    pairs_by_account = pairs.groupby("account")
    accounts["n_distinct_counterparties"] = pairs_by_account.size()
    accounts["max_pair_multiplicity"] = pairs_by_account["n_payments"].max()
    accounts["counterparty_entropy"] = pairs_by_account["entropy_term"].sum()
    accounts["mean_counterparty_indegree"] = pairs_by_account["indegree"].mean()
    accounts["frac_shared_counterparties"] = pairs_by_account["counterparty_is_shared"].mean()
    accounts["repeat_rate"] = 1 - accounts["n_distinct_counterparties"] / accounts[
        "n_counterparty_transactions"].where(accounts["n_counterparty_transactions"] > 0)

    account_bank_counts = (edges.groupby(["account", "counterparty_bank"]).size()
                           .rename("n_transactions").reset_index())
    account_bank_counts["share"] = account_bank_counts["n_transactions"] / account_bank_counts.groupby(
        "account")["n_transactions"].transform("sum")
    account_bank_counts["entropy_term"] = -account_bank_counts["share"] * np.log(
        account_bank_counts["share"])
    accounts["n_banks"] = account_bank_counts.groupby("account").size()
    accounts["bank_entropy"] = account_bank_counts.groupby("account")["entropy_term"].sum()
    accounts["median_gap_days"] = edges.groupby("account")["days_since_previous_payment"].median()
    accounts_with_counterparty = accounts[accounts["n_counterparty_transactions"] > 0]

    # bipartite account-counterparty graph ------------------------------------
    account_idx, account_labels = pd.factorize(pairs["account"])
    counterparty_idx, counterparty_labels = pd.factorize(pairs["counterparty"])
    n_accounts, n_counterparties = len(account_labels), len(counterparty_labels)
    adjacency = bipartite_adjacency(account_idx, counterparty_idx, n_accounts, n_counterparties)
    components = connected_component_stats(adjacency)
    n_butterflies, n_wedges = count_butterflies_and_wedges(adjacency)
    n_pairs = len(pairs)

    # bank co-usage (account x bank) ------------------------------------------
    bank_codes, bank_labels = pd.factorize(account_bank_counts["counterparty_bank"])
    account_codes, _ = pd.factorize(account_bank_counts["account"])
    account_bank_matrix = sparse.csr_matrix(
        (np.ones(len(account_bank_counts)), (account_codes, bank_codes)))
    bank_cousage = pd.DataFrame(
        (account_bank_matrix.T @ account_bank_matrix).toarray() / account_bank_matrix.shape[0],
        index=bank_labels, columns=bank_labels)

    # temporal ------------------------------------------------------------------
    gaps = edges["days_since_previous_payment"].dropna()
    pair_gap_stats = (edges.dropna(subset=["days_since_previous_payment"])
                      .groupby(pair_keys)["days_since_previous_payment"]
                      .agg(["count", "mean", "std"]))
    pair_gap_stats = pair_gap_stats[(pair_gap_stats["count"] >= 2) & (pair_gap_stats["mean"] > 0)]
    gap_variation_per_pair = (pair_gap_stats["std"] / pair_gap_stats["mean"]).dropna()
    nearest_month = np.round(gaps / 30.0)
    monthly_gap_share = (
        float(((nearest_month >= 1) & ((gaps - 30 * nearest_month).abs() <= 3)).mean())
        if len(gaps) >= MIN_SAMPLE_SIZE else np.nan)

    # recurring-payment consistency: attributes should stay stable within a repeated pair
    repeated_pairs = edges.groupby(pair_keys, sort=False).agg(
        n_payments=("counterparty", "size"),
        n_distinct_k_symbols=("k_symbol", "nunique"),
        n_distinct_operations=("operation", "nunique"),
        n_distinct_types=("trans_type", "nunique"),
        log_amount_sd=("log_amount", "std"))
    repeated_pairs = repeated_pairs[repeated_pairs["n_payments"] >= 2]
    enough_repeated_pairs = len(repeated_pairs) >= MIN_SAMPLE_SIZE

    def share_with_single_value(column):
        return float((repeated_pairs[column] == 1).mean()) if enough_repeated_pairs else np.nan

    # a counterparty account number should belong to a single bank
    multi_bank_share = float(
        (edges.groupby("counterparty_account_number")["counterparty_bank"].nunique() > 1).mean())

    counterparty_volume = counterparty_stats["n_transactions"].sort_values(ascending=False)
    n_top_counterparties = max(1, int(round(0.01 * len(counterparty_volume))))

    metrics = {}

    def register(group, name, kind, payload):
        metrics[(group, name)] = (kind, payload)

    register("presence", "cp_presence_rate", "rate", float(transactions["has_counterparty"].mean()))
    register("presence", "accounts_with_any_cp", "rate",
             float((accounts["n_counterparty_transactions"] > 0).mean()))
    register("presence", "cp_share_per_account", "ks", accounts["counterparty_share"].values)

    register("account", "n_tx", "ks", accounts["n_transactions"].values)
    register("account", "n_cp_tx", "ks", accounts["n_counterparty_transactions"].values)
    for metric_name, column in ACCOUNT_METRIC_COLUMNS.items():
        register("account", metric_name, "ks", accounts_with_counterparty[column].dropna().values)

    register("pair", "edge_multiplicity", "ks", pairs["n_payments"].values)
    register("pair", "repeat_tx_share", "rate", float(edges["is_repeat_payment"].mean()))

    register("counterparty", "cp_indegree", "ks", counterparty_stats["indegree"].values)
    register("counterparty", "cp_tx_count", "ks", counterparty_stats["n_transactions"].values)
    register("counterparty", "tx_to_shared_cp_share", "rate",
             float((edges["counterparty_indegree"] > 1).mean()))
    register("counterparty", "top1pct_cp_tx_share", "rate",
             float(counterparty_volume.iloc[:n_top_counterparties].sum() / counterparty_volume.sum()))
    register("counterparty", "acct_number_multi_bank_share", "rate", multi_bank_share)

    register("edge_consistency", "repeated_edge_k_symbol_purity", "rate",
             share_with_single_value("n_distinct_k_symbols"))
    register("edge_consistency", "repeated_edge_operation_purity", "rate",
             share_with_single_value("n_distinct_operations"))
    register("edge_consistency", "repeated_edge_type_purity", "rate",
             share_with_single_value("n_distinct_types"))
    register("edge_consistency", "repeated_edge_log_amount_sd", "ks",
             repeated_pairs["log_amount_sd"].dropna().values if enough_repeated_pairs else np.array([]))

    register("structure", "giant_component_frac", "rate", components["giant_component_fraction"])
    register("structure", "dyad_component_share", "rate", components["dyad_component_share"])
    register("structure", "wedges_per_edge", "logratio", n_wedges / n_pairs)
    register("structure", "butterflies_per_edge", "logratio", n_butterflies / n_pairs)

    register("bank", "bank_mix", "tv", edges["counterparty_bank"].value_counts())
    register("bank", "banks_per_account", "ks", accounts_with_counterparty["n_banks"].dropna().values)
    register("bank", "bank_cousage_matrix", "matrix", bank_cousage)

    register("temporal", "gap_days", "ks", gaps.values if len(gaps) >= MIN_SAMPLE_SIZE else np.array([]))
    register("temporal", "gap_cv_per_pair", "ks",
             gap_variation_per_pair.values if len(gap_variation_per_pair) >= MIN_SAMPLE_SIZE
             else np.array([]))
    register("temporal", "monthly_gap_share", "rate", monthly_gap_share)
    register("temporal", "date_parse_fail_rate", "rate", float(transactions["date"].isna().mean()))

    return dict(
        transactions=transactions,
        edges=edges,
        accounts=accounts,
        metrics=metrics,
        graph_indices=(account_idx, counterparty_idx, n_accounts, n_counterparties),
        n_butterflies=n_butterflies,
        n_pairs=n_pairs,
        shared_counterparty_share=float((counterparty_stats["indegree"] > 1).mean()),
    )


# --------------------------------------------------------------------------- #
# Distances
# --------------------------------------------------------------------------- #
DISTANCE_DESCRIPTIONS = {
    "ks": (
        "KS statistic",
        "Largest vertical gap between the two cumulative distribution curves. "
        "Compares the whole distribution, not just the mean. 0 = identical, 1 = no overlap."),
    "tv": (
        "Total variation",
        "Half the sum of absolute differences between category shares (the share of "
        "probability mass that would have to move). Compares the whole categorical "
        "distribution. 0 = identical, 1 = disjoint."),
    "rate": (
        "Absolute difference",
        "abs(real - synthetic) of one number (a rate, share or score). The metric is already "
        "a single value, so there is no distribution to compare. 0 = identical."),
    "logratio": (
        "Absolute log-ratio",
        "abs(log((real + eps) / (synthetic + eps))) for an unbounded non-negative count. "
        "Scale-free: 0 = equal, 0.69 = a factor of 2 apart."),
    "matrix": (
        "Mean absolute matrix difference",
        "Mean over all cells of abs(real - synthetic) for the bank co-usage matrix "
        "(share of accounts using both bank i and bank j). Compares the full pairwise "
        "structure cell by cell. 0 = identical."),
    "wabs": (
        "Weighted absolute rate difference",
        "Per category, abs(real rate - synthetic rate) of having a counterparty, averaged with "
        "weights = real category size. For a yes/no variable the rate is its whole "
        "distribution, so each category is compared exactly. 0 = identical."),
    "weighted_ks": (
        "KS statistic per bucket, weighted",
        "KS statistic within each graph-position bucket, averaged with weights = real bucket "
        "size. A bucket that exists in real but is (almost) empty in synthetic counts as 1."),
    "weighted_tv": (
        "Total variation per bucket, weighted",
        "Total variation within each graph-position bucket, averaged with weights = real bucket "
        "size. A bucket that exists in real but is (almost) empty in synthetic counts as 1."),
    "auc": (
        "Classifier AUC distance",
        "abs(AUC - 0.5) of a gradient-boosted classifier separating real from synthetic accounts "
        "using all per-account graph features jointly (so joint distributions, not just "
        "marginals, matter). 0 = indistinguishable, 0.5 = perfectly separable."),
}


def _total_variation(counts_a, counts_b):
    """Total variation distance between two category-count Series."""
    p = counts_a / counts_a.sum()
    q = counts_b / counts_b.sum()
    categories = p.index.union(q.index)
    return 0.5 * float(
        (p.reindex(categories).fillna(0) - q.reindex(categories).fillna(0)).abs().sum())


def _distance(kind, real_payload, synthetic_payload):
    if kind == "ks":
        if len(real_payload) == 0 and len(synthetic_payload) == 0:
            return 0.0
        if len(real_payload) == 0 or len(synthetic_payload) == 0:
            return 1.0  # quantity exists on only one side
        return float(ks_2samp(real_payload, synthetic_payload).statistic)
    if kind == "tv":
        return _total_variation(real_payload, synthetic_payload)
    if kind in ("rate", "logratio"):
        real_missing, synthetic_missing = np.isnan(real_payload), np.isnan(synthetic_payload)
        if real_missing and synthetic_missing:
            return 0.0
        if real_missing or synthetic_missing:
            return 1.0  # quantity exists on only one side
        if kind == "rate":
            return abs(real_payload - synthetic_payload)
        return abs(float(np.log((real_payload + LOG_RATIO_EPSILON)
                                / (synthetic_payload + LOG_RATIO_EPSILON))))
    if kind == "matrix":
        labels = real_payload.index.union(synthetic_payload.index)
        real_values = real_payload.reindex(index=labels, columns=labels).fillna(0).values
        synthetic_values = synthetic_payload.reindex(index=labels, columns=labels).fillna(0).values
        return float(np.abs(real_values - synthetic_values).mean())
    raise ValueError(kind)


def _summary_value(kind, payload):
    """Scalar shown in the `real` / `synthetic` columns (mean for distributions)."""
    if kind == "ks":
        return float(np.mean(payload)) if len(payload) else np.nan
    if kind in ("rate", "logratio"):
        return float(payload)
    return np.nan


def compare_single_table_metrics(real_bundle, synthetic_bundle):
    """Rows (group, metric, kind, real, synthetic, distance) for metrics computed per table."""
    rows = []
    for (group, name), (kind, real_payload) in real_bundle["metrics"].items():
        synthetic_payload = synthetic_bundle["metrics"][(group, name)][1]
        rows.append((group, name, kind,
                     _summary_value(kind, real_payload), _summary_value(kind, synthetic_payload),
                     _distance(kind, real_payload, synthetic_payload)))
    return rows


def _assign_bucket(edges, bucketing):
    if bucketing == "pair_multiplicity":
        return pd.cut(edges["pair_multiplicity"], [0, 1, 3, np.inf],
                      labels=["1", "2-3", "4+"]).astype(str)
    return pd.cut(edges["counterparty_indegree"], [0, 1, np.inf], labels=["1", "2+"]).astype(str)


def compare_conditional_metrics(real_bundle, synthetic_bundle):
    """Metrics that need both tables at once (presence by category, coupling)."""
    rows = []
    real_transactions = real_bundle["transactions"]
    synthetic_transactions = synthetic_bundle["transactions"]

    for label, column in PRESENCE_CATEGORIES.items():
        real_stats = real_transactions.groupby(column)["has_counterparty"].agg(["mean", "size"])
        synthetic_rate = (synthetic_transactions.groupby(column)["has_counterparty"].mean()
                          .reindex(real_stats.index).fillna(0))
        weights = real_stats["size"] / real_stats["size"].sum()
        rows.append(("presence", f"cp_presence_by_{label}", "wabs", np.nan, np.nan,
                     float((weights * (real_stats["mean"] - synthetic_rate).abs()).sum())))

    real_edges, synthetic_edges = real_bundle["edges"], synthetic_bundle["edges"]
    for bucketing_label, bucketing in COUPLING_BUCKETINGS.items():
        real_bucket = _assign_bucket(real_edges, bucketing)
        synthetic_bucket = _assign_bucket(synthetic_edges, bucketing)
        for variable_label, variable in COUPLING_VARIABLES.items():
            weighted_distance_sum, total_weight = 0.0, 0.0
            for bucket, n_real_rows in real_bucket.value_counts().items():
                real_rows = real_edges[real_bucket == bucket]
                synthetic_rows = synthetic_edges[synthetic_bucket == bucket]
                if len(real_rows) < MIN_SAMPLE_SIZE:
                    continue
                if len(synthetic_rows) < MIN_SAMPLE_SIZE:
                    distance = 1.0  # bucket exists in real data but (almost) not in synthetic
                elif variable == "log_amount":
                    distance = ks_2samp(real_rows[variable].dropna(),
                                        synthetic_rows[variable].dropna()).statistic
                else:
                    distance = _total_variation(real_rows[variable].value_counts(),
                                                synthetic_rows[variable].value_counts())
                weighted_distance_sum += n_real_rows * distance
                total_weight += n_real_rows
            rows.append(("coupling", f"{variable_label} | {bucketing_label} bucket",
                         "weighted_ks" if variable == "log_amount" else "weighted_tv", np.nan, np.nan,
                         weighted_distance_sum / total_weight if total_weight else np.nan))
    return rows


# --------------------------------------------------------------------------- #
# Null-model calibration and classifier two-sample test
# --------------------------------------------------------------------------- #
def butterfly_null_calibration(bundle, n_null_graphs, swaps_per_edge, rng):
    """Observed butterflies relative to degree-preserving random rewirings."""
    account_idx, counterparty_idx, n_accounts, n_counterparties = bundle["graph_indices"]
    null_counts = []
    for _ in range(n_null_graphs):
        rewired = degree_preserving_edge_swaps(
            account_idx, counterparty_idx, int(swaps_per_edge * len(account_idx)), rng)
        null_adjacency = bipartite_adjacency(account_idx, rewired, n_accounts, n_counterparties)
        null_counts.append(count_butterflies_and_wedges(null_adjacency)[0])
    null_counts = np.asarray(null_counts, dtype=float)
    observed = bundle["n_butterflies"]
    null_sd = null_counts.std(ddof=1) if len(null_counts) > 1 else np.nan
    z_score = (observed - null_counts.mean()) / null_sd if null_sd and null_sd > 0 else np.nan
    excess_ratio = (observed + 1) / (null_counts.mean() + 1)
    return excess_ratio, z_score


def classifier_two_sample_test(features_a, features_b, seed=0, n_folds=5):
    """Cross-validated AUC of a classifier separating the two sets of account features."""
    n = min(len(features_a), len(features_b))
    sample_a = features_a.sample(n, random_state=seed)
    sample_b = features_b.sample(n, random_state=seed)
    X = pd.concat([sample_a, sample_b])[CLASSIFIER_FEATURES]
    X = X.loc[:, X.notna().sum() >= 20]  # drop (almost) empty features
    y = np.r_[np.zeros(n), np.ones(n)]
    classifier = HistGradientBoostingClassifier(max_depth=3, max_iter=100, random_state=seed)
    folds = StratifiedKFold(n_folds, shuffle=True, random_state=seed)
    predicted = cross_val_predict(classifier, X, y, cv=folds, method="predict_proba")[:, 1]
    return float(roc_auc_score(y, predicted))


# --------------------------------------------------------------------------- #
# Orchestration
# --------------------------------------------------------------------------- #
def evaluate(real_trans, synthetic_trans, column_map=COLUMN_MAP, n_null_graphs=3,
             swaps_per_edge=5, n_noise_floor_splits=3, seed=0,
             min_shared_counterparty_share_for_motifs=0.05):
    """Compare `synthetic_trans` to `real_trans`; returns one row per metric with
    columns group, metric, distance_measure, distance_explanation, real, synthetic,
    distance, noise_floor, ratio, flag."""
    rng = np.random.default_rng(seed)
    real_trans, synthetic_trans = match_account_counts(
        real_trans, synthetic_trans, column_map, seed)

    real_bundle = build_metric_bundle(real_trans, column_map)
    synthetic_bundle = build_metric_bundle(synthetic_trans, column_map)
    result_columns = ["group", "metric", "kind", "real", "synthetic", "distance"]
    rows = (compare_single_table_metrics(real_bundle, synthetic_bundle)
            + compare_conditional_metrics(real_bundle, synthetic_bundle))

    # Wedge / butterfly metrics only carry information when counterparties are shared
    # between accounts in the REAL data (otherwise the graph is a forest of stars).
    use_motifs = (real_bundle["shared_counterparty_share"]
                  >= min_shared_counterparty_share_for_motifs)
    if use_motifs:
        real_excess, real_z = butterfly_null_calibration(
            real_bundle, n_null_graphs, swaps_per_edge, rng)
        synthetic_excess, synthetic_z = butterfly_null_calibration(
            synthetic_bundle, n_null_graphs, swaps_per_edge, rng)
        rows.append(("structure", "butterfly_excess_vs_null (obs+1)/(null+1)", "logratio",
                     real_excess, synthetic_excess,
                     _distance("logratio", real_excess, synthetic_excess)))
        rows.append(("structure", "butterfly_z_vs_null", "rate", real_z, synthetic_z,
                     abs(real_z - synthetic_z)
                     if np.isfinite(real_z) and np.isfinite(synthetic_z) else np.nan))

    # Noise floor: the same distances between two random halves of the real accounts.
    noise_floor_frames, noise_floor_aucs = [], []
    for split in range(n_noise_floor_splits):
        half_a, half_b = split_accounts_in_half(real_trans, column_map, seed + split)
        bundle_a = build_metric_bundle(half_a, column_map)
        bundle_b = build_metric_bundle(half_b, column_map)
        noise_floor_frames.append(pd.DataFrame(
            compare_single_table_metrics(bundle_a, bundle_b)
            + compare_conditional_metrics(bundle_a, bundle_b), columns=result_columns))
        noise_floor_aucs.append(classifier_two_sample_test(
            bundle_a["accounts"], bundle_b["accounts"], seed + split))
    noise_floor = (pd.concat(noise_floor_frames).groupby(["group", "metric"])["distance"]
                   .mean().rename("noise_floor"))

    auc_metric_name = "account_graph_features_AUC (0.5 = indistinguishable)"
    auc = classifier_two_sample_test(
        real_bundle["accounts"], synthetic_bundle["accounts"], seed)
    rows.append(("c2st", auc_metric_name, "auc", auc, np.nan, abs(auc - 0.5)))
    noise_floor = pd.concat([noise_floor, pd.Series(
        {("c2st", auc_metric_name): float(np.mean([abs(x - 0.5) for x in noise_floor_aucs]))},
        name="noise_floor")])

    results = pd.DataFrame(rows, columns=result_columns).join(
        noise_floor, on=["group", "metric"])
    results["ratio"] = results["distance"] / results["noise_floor"].clip(lower=MIN_NOISE_FLOOR)
    results["flag"] = np.where(
        results["ratio"].isna(), "",
        np.where(results["ratio"] <= GAP_RATIO_THRESHOLD, "ok", "GAP"))
    results["distance_measure"] = results["kind"].map(
        {kind: name for kind, (name, _) in DISTANCE_DESCRIPTIONS.items()})
    results["distance_explanation"] = results["kind"].map(
        {kind: text for kind, (_, text) in DISTANCE_DESCRIPTIONS.items()})
    results = results.drop(columns="kind")[
        ["group", "metric", "distance_measure", "distance_explanation", "real", "synthetic",
         "distance", "noise_floor", "ratio", "flag"]]
    if not use_motifs:
        results = results[~results["metric"].isin(["wedges_per_edge", "butterflies_per_edge"])]
    group_order = ["presence", "account", "pair", "counterparty", "edge_consistency",
                   "structure", "bank", "temporal", "coupling", "c2st"]
    results["group"] = pd.Categorical(results["group"], group_order, ordered=True)
    return results.sort_values("group", kind="stable").reset_index(drop=True)


In [3]:
import os
from pathlib import Path


def set_project_root(marker="pyproject.toml"):
    """Change the working directory to the repository root."""
    path = Path.cwd()
    for parent in [path, *path.parents]:
        if (parent / marker).exists():
            os.chdir(parent)
            return parent
    raise FileNotFoundError(f"Could not find {marker} in any parent directory")


set_project_root()

PosixPath('/home/coder/vector-bootcamp-synthetic-data')

In [4]:
def drop_aux_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Drop placeholder and cluster columns added during synthesis."""
    drop = [col for col in df.columns if col == "placeholder" or col.endswith("_cluster")]
    return df.drop(columns=drop, errors="ignore")

In [5]:
import pickle

ROOT = Path.cwd()
IMPLEMENTATION_ROOT = ROOT / "implementations" / "tabular_data"
base_data_dir = IMPLEMENTATION_ROOT / "multi_table" / "data" / "berka"
base_output_dir = IMPLEMENTATION_ROOT / "multi_table" / "results"

PARENT_TABLE = "account"

pickle_path = base_output_dir / "before_matching" / "synthetic_tables.pkl"
with open(pickle_path, "rb") as f:
    synthetic_tables = pickle.load(f)

real_trans = pd.read_csv(base_data_dir / "trans.csv")
synthetic_trans = drop_aux_columns(synthetic_tables[(PARENT_TABLE, "trans")]["df"])

# real_trans = pd.read_csv(REAL_TRANS_PATH)
# synthetic_trans = pd.read_csv(SYNTHETIC_TRANS_PATH)

results = evaluate(real_trans, synthetic_trans)
with pd.option_context("display.max_rows", None, "display.width", 200,
                        "display.max_colwidth", 70, "display.float_format", "{:.4f}".format):
    print(results.drop(columns="distance_explanation").to_string())
    print("\nGAP count per group:")
    print(results.assign(gap=results["flag"].eq("GAP"), n=results["flag"].ne(""))
            .groupby("group", observed=True)[["gap", "n"]].sum())
results.to_csv(OUT_PATH, index=False)

/home/coder/vector-bootcamp-synthetic-data/.venv/lib/python3.12/site-packages/scipy/stats/_axis_nan_policy.py:579: RuntimeWarning: ks_2samp: Exact calculation unsuccessful. Switching to method=asymp.
  res = hypotest_fun_out(*samples, **kwds)


               group                                                metric                      distance_measure     real  synthetic  distance  noise_floor     ratio flag
0           presence                                      cp_presence_rate                   Absolute difference   0.2584     0.2449    0.0135       0.0014    9.8954  GAP
1           presence                                  accounts_with_any_cp                   Absolute difference   0.7858     0.9538    0.1680       0.0010  166.9773  GAP
2           presence                                  cp_share_per_account                          KS statistic   0.2467     0.2450    0.1750       0.0204    8.5712  GAP
3           presence                                   cp_presence_by_type     Weighted absolute rate difference      NaN        NaN    0.0172       0.0025    6.9945  GAP
4           presence                              cp_presence_by_operation     Weighted absolute rate difference      NaN        NaN    0.0106   